# 💬 Chat Message Intent Classification using NLP

> **Goal:** Classify customer/chat support messages into 3 intent categories:  
> `Support` | `Complaint` | `Query`

**Tech Stack:** Python · Scikit-learn · Pandas · NLTK · TF-IDF · Logistic Regression

---

## 📦 1. Import Libraries

In [ ]:
# Standard libraries
import re
import os
import sys
import string
import warnings
warnings.filterwarnings('ignore')

# Data handling
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# NLP
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

# Machine Learning
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay
)
import joblib

print('All libraries imported successfully ✅')

## 📂 2. Load Dataset

In [ ]:
# Load the CSV dataset
df = pd.read_csv('../data/intents_dataset.csv')

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(10)

In [ ]:
# Label distribution
print('Label Distribution:')
print(df['label'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
colors = ['#4C72B0', '#DD8452', '#55A868']
df['label'].value_counts().plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_title('Label Distribution (Bar Chart)', fontweight='bold')
axes[0].set_xlabel('Intent Label')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=0)

# Pie chart
df['label'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%',
                                colors=colors, startangle=90)
axes[1].set_title('Label Distribution (Pie Chart)', fontweight='bold')
axes[1].set_ylabel('')

plt.tight_layout()
plt.show()

## 🔧 3. Text Preprocessing

In [ ]:
def preprocess_text(text: str) -> str:
    """Full NLP preprocessing pipeline."""
    # Step 1: Lowercase
    text = text.lower()
    # Step 2: Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Step 3: Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Step 4: Tokenize
    tokens = word_tokenize(text)
    # Step 5: Remove stopwords
    stop_words = set(stopwords.words('english'))
    tokens = [w for w in tokens if w not in stop_words]
    return ' '.join(tokens)

# Apply preprocessing
df['cleaned_message'] = df['message'].apply(preprocess_text)

# Show before/after comparison
comparison = df[['message', 'cleaned_message', 'label']].head(6)
comparison.columns = ['Original Message', 'Cleaned Message', 'Label']
comparison

## 📊 4. Exploratory Data Analysis (EDA)

In [ ]:
# Message length analysis
df['msg_length']     = df['message'].apply(len)
df['word_count']     = df['message'].apply(lambda x: len(x.split()))
df['cleaned_length'] = df['cleaned_message'].apply(len)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Word count distribution per label
for label, color in zip(['support', 'complaint', 'query'], colors):
    subset = df[df['label'] == label]['word_count']
    axes[0].hist(subset, bins=10, alpha=0.6, label=label, color=color)
axes[0].set_title('Word Count Distribution by Label', fontweight='bold')
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')
axes[0].legend()

# Average word count per label
avg_words = df.groupby('label')['word_count'].mean()
avg_words.plot(kind='bar', ax=axes[1], color=colors, edgecolor='black')
axes[1].set_title('Average Word Count per Label', fontweight='bold')
axes[1].set_xlabel('Label')
axes[1].set_ylabel('Average Word Count')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.show()

print(df.groupby('label')[['word_count', 'msg_length']].describe().round(2))

## ✂️ 5. Train-Test Split

In [ ]:
X = df['cleaned_message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples : {len(X_train)} ({len(X_train)/len(X)*100:.0f}%)')
print(f'Testing samples  : {len(X_test)}  ({len(X_test)/len(X)*100:.0f}%)')
print(f'\nTrain label dist :\n{y_train.value_counts()}')
print(f'\nTest label dist  :\n{y_test.value_counts()}')

## 🔢 6. TF-IDF Vectorization

In [ ]:
# Create and fit TF-IDF vectorizer
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'Vocabulary size       : {len(tfidf.vocabulary_)}')
print(f'Training matrix shape : {X_train_tfidf.shape}')
print(f'Testing matrix shape  : {X_test_tfidf.shape}')

# Show top 15 TF-IDF terms
feature_names = tfidf.get_feature_names_out()
print(f'\nSample TF-IDF features: {list(feature_names[:15])}')

## 🤖 7. Train Logistic Regression

In [ ]:
# Train the model
model = LogisticRegression(max_iter=1000, solver='lbfgs', C=1.0, random_state=42)
model.fit(X_train_tfidf, y_train)

print('Model trained successfully ✅')
print(f'Classes : {list(model.classes_)}')

# Cross-validation for robust accuracy estimate
from sklearn.pipeline import Pipeline
pipeline = Pipeline([('tfidf', TfidfVectorizer(max_features=5000, ngram_range=(1,2), sublinear_tf=True)),
                     ('clf', LogisticRegression(max_iter=1000, random_state=42))])
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
print(f'\n5-Fold Cross-Validation Accuracy:')
print(f'  Scores : {[f"{s:.2f}" for s in cv_scores]}')
print(f'  Mean   : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')

## 📈 8. Evaluate Model

In [ ]:
# Predictions
y_pred = model.predict(X_test_tfidf)
acc    = accuracy_score(y_test, y_pred)

print(f'Test Accuracy: {acc * 100:.2f}%\n')
print('Classification Report:')
print(classification_report(y_test, y_pred, target_names=model.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=model.classes_, yticklabels=model.classes_,
            linewidths=0.5, annot_kws={'size': 14, 'weight': 'bold'})
plt.title('Confusion Matrix — Intent Classification', fontsize=14, fontweight='bold', pad=12)
plt.ylabel('Actual Label', fontsize=12)
plt.xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print('Confusion matrix saved to outputs/confusion_matrix.png')

## 🧠 9. Feature Importance (Top TF-IDF Features per Class)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, (cls, color) in enumerate(zip(model.classes_, colors)):
    # Get coefficients for this class
    coef = model.coef_[idx]
    top_indices = np.argsort(coef)[-12:][::-1]
    top_features = [feature_names[i] for i in top_indices]
    top_scores   = [coef[i] for i in top_indices]
    
    axes[idx].barh(top_features[::-1], top_scores[::-1], color=color, edgecolor='black', alpha=0.8)
    axes[idx].set_title(f'Top Features: {cls.upper()}', fontweight='bold')
    axes[idx].set_xlabel('Coefficient Weight')

plt.suptitle('Most Important TF-IDF Features per Intent Class', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 💾 10. Save Model & Vectorizer

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/intent_model.pkl')
joblib.dump(tfidf, '../models/vectorizer.pkl')
print('Model saved to      ../models/intent_model.pkl ✅')
print('Vectorizer saved to ../models/vectorizer.pkl   ✅')

## 🔮 11. Predict on New Messages

In [ ]:
def predict_intent(message: str) -> dict:
    """Predict intent label and confidence for a raw message."""
    cleaned  = preprocess_text(message)
    vector   = tfidf.transform([cleaned])
    label    = model.predict(vector)[0]
    proba    = model.predict_proba(vector)[0]
    proba_d  = {cls: round(p * 100, 1) for cls, p in zip(model.classes_, proba)}
    return {'label': label, 'confidence': proba_d[label], 'probabilities': proba_d}

# Test predictions
test_messages = [
    ('I need help resetting my password',              'support'),
    ('This service is absolutely terrible!',            'complaint'),
    ('What are the available subscription plans?',      'query'),
    ('Please guide me through the account setup',       'support'),
    ('I am very upset about the billing error',         'complaint'),
    ('Do you offer a student discount?',                'query'),
]

print(f'{"Message":<52} {"Actual":<12} {"Predicted":<12} {"Confidence"}')
print('-' * 90)

correct = 0
for msg, actual in test_messages:
    result = predict_intent(msg)
    tick   = '✓' if result['label'] == actual else '✗'
    correct += (result['label'] == actual)
    print(f"{msg[:50]:<52} {actual:<12} {result['label']:<12} {result['confidence']:.1f}% {tick}")

print(f'\nSample Accuracy: {correct}/{len(test_messages)} ({correct/len(test_messages)*100:.0f}%)')

---
## ✅ Summary

| Step | Description | Status |
|------|-------------|--------|
| 1 | Load Dataset (90 balanced messages) | ✅ |
| 2 | NLP Preprocessing (lowercase, stopwords, tokenize) | ✅ |
| 3 | TF-IDF Vectorization (unigrams + bigrams) | ✅ |
| 4 | Train-Test Split (80/20, stratified) | ✅ |
| 5 | Logistic Regression Training | ✅ |
| 6 | Evaluation (Accuracy + Confusion Matrix) | ✅ |
| 7 | Save Model + Vectorizer | ✅ |
| 8 | Prediction System | ✅ |